# IDMapping：因子库证券代码转换

不同数据源可能使用不同的证券代码。`IDMappingFactorDB` 通过包装已有 `FactorDB`，在不修改原生因子库的情况下完成双向转换：调用方传入外部 ID，包装层先转换成原生 ID 读取数据，再把结果的 ID 转回外部命名空间。

```mermaid
flowchart LR
    A[调用方外部 ID] -->|resolve| B[原生数据源 ID]
    B --> C[原生 FactorDB / FactorTable]
    C --> D[原生因子数据]
    D -->|反向投影| E[外部 ID 因子数据]
```

## 核心对象

| 对象 | 作用 |
|---|---|
| `IDMapping` | 映射抽象，定义外部 ID → 原生 ID，以及反向转换 |
| `StaticIDMapping` | 使用内存字典构造映射，适合测试和固定规则 |
| `FactorDBIDMappingConfig` | 配置默认映射、按表覆盖和证券代码值字段 |
| `IDMappingFactorDB` | 包装原生 `FactorDB` |
| `IDMappingFactorTable` | 包装原生 `FactorTable`，负责读取前后的转换 |

In [1]:
import datetime as dt

from QSExt.Factor.IDMapping import (
    FactorDBIDMappingConfig,
    IDMappingFactorDB,
    StaticIDMapping,
)

## 静态映射示例

映射字典的方向固定为 `{外部 ID: 原生 ID}`。允许多个外部 ID 指向同一个原生 ID；请求原生数据时会自动去重，返回时再展开到原始外部 ID。

In [2]:
mapping = StaticIDMapping(args={
    "Name": "示例代码映射",
    "Mapping": {
        "EXT-A": "NATIVE-1",
        "EXT-B": "NATIVE-1",
        "EXT-C": "NATIVE-2",
    },
    "UnmappedPolicy": "empty",
})

request = mapping.resolve(["EXT-A", "EXT-B", "UNKNOWN"])
print(request.native_ids)          # ['NATIVE-1']
print(request.external_to_native)  # ['NATIVE-1', 'NATIVE-1', None]
print(mapping.MappingID)           # 映射内容的稳定摘要

['NATIVE-1']
['NATIVE-1', 'NATIVE-1', None]
96ebd884e79e595c3c2a5301cf7b9b8394f4270c9fb1b0c7f8dfdbde94cda191


### 未映射 ID 策略

`UnmappedPolicy` 控制映射中不存在的 ID：

- `empty`：不向原生库请求该 ID，输出为空值；
- `raise`：立即抛出异常，适合严格质量控制；
- `passthrough`：原样传递，适合两套命名空间中部分代码本来就一致的场景。

`MappingID` 包含映射内容和策略。当映射关系变化时，QSID 和因子缓存键也会随之变化。

## 包装任意 FactorDB

下面使用 `AKShareDB` 演示通用包装流程。假设研究侧使用不补前导零的港股代码 `700.HK`、`2318.HK`，而 AKShareDB 使用 `00700.HK`、`02318.HK`。

`Default` 应用于没有单独配置的表；`Tables` 可以按精确表名替换映射或使用 `None` 关闭映射。包装层只依赖标准 `FactorDB` / `FactorTable` 接口，因此同样适用于其他原生因子库。

In [22]:
from QSExt.Factor.AKShareDB import AKShareDB

AKSFDB = AKShareDB().connect()
aks_mapping = StaticIDMapping(args={
    "Name": "研究代码到 AKShare 港股代码",
    "Mapping": {
        "700.HK": "00700",
        "2318.HK": "02318",
    },
    "UnmappedPolicy": "raise",
})
config = FactorDBIDMappingConfig(Default=aks_mapping)
FDB = IDMappingFactorDB(
    AKSFDB, args={"IDMappingConfig": config}
)

FT = FDB.getTable(
    "港股-历史行情数据-新浪",
    args={"SourceArgs": {"LookBack": 0}},
)
DTs = [dt.datetime(2026, 11, 9), dt.datetime(2026, 11, 10)]
Data = FT.readData(
    factor_names=["high", "close"],
    ids=["700.HK", "2318.HK"],
    dts=DTs,
)
print(Data.iloc[0])

            700.HK  2318.HK
2026-11-09   428.4     53.6
2026-11-10   424.4     53.4


### 获取表时覆盖配置

包装层保留以下表级参数：

| 参数 | 说明 |
|---|---|
| `IDMapping` | 覆盖当前表的映射；设置为 `None` 可关闭映射 |
| `IDValueMappings` | 当前表中因子名称到对应 `IDMapping` 的映射 |
| `IDValueAmbiguity` | 因子值一对多反向映射策略 |
| `SourceArgs` | 传递给原生 `FactorDB.getTable` 的参数 |

`SourceArgs` 用于隔离包装层参数与原生表参数，尤其适合参数名称发生冲突时。

In [26]:
MappedFT = FDB.getTable(
    "港股-历史行情数据-新浪",
    args={
        "IDMapping": aks_mapping,
        "IDValueMappings": {},
        "IDValueAmbiguity": "raise",
        "SourceArgs": {"LookBack": 0},
    },
)
MappedData = MappedFT.readData(
    factor_names=["close"],
    ids=["700.HK"],
    dts=DTs,
)

NativeFT = FDB.getTable(
    "港股-历史行情数据-新浪",
    args={
        "IDMapping": None,
        "SourceArgs": {"LookBack": 0},
    },
)
NativeData = NativeFT.readData(
    factor_names=["close"],
    ids=["00700"],
    dts=DTs,
)
print(MappedData.iloc[0])
print(NativeData.iloc[0])

               700.HK
2026-11-09  418.39999
2026-11-10  417.79999
                00700
2026-11-09  418.39999
2026-11-10  417.79999


## 转换因子字段中的证券代码

有些因子的值本身也是证券代码，例如关联证券、成分证券或基金主代码。通过 `IDValueMappings={表名: {因子名: IDMapping}}` 为每个字段选择映射，可以在行列 ID 转换后继续转换这些单元格；同一张表的不同字段可以使用不同映射。支持单个字符串以及 `list`、`tuple`、`set` 容器。

一个原生 ID 可能对应多个外部 ID，例如证券发生代码迁移。`IDValueAmbiguity` 的取值为：

- `raise`：存在多个候选时抛出异常，默认且最安全；
- `first`：使用映射顺序中的第一个外部 ID；
- `list`：将全部外部 ID 作为列表写入单元格。

In [28]:
value_mapping = StaticIDMapping(args={
    "Name": "AKShare 港股行列及字段值映射",
    "Mapping": {
        "700.HK": "00700",
        "2318.HK": "02318",
        "700": "00700",
        "2318": "02318",
    },
    "UnmappedPolicy": "passthrough",
})
value_config = FactorDBIDMappingConfig(
    Default=value_mapping,
    IDValueMappings={
        "港股-证券资料-东财": {"证券代码": value_mapping},
    },
    IDValueAmbiguity="raise",
)
ValueFDB = IDMappingFactorDB(
    AKSFDB, args={"IDMappingConfig": value_config}
)
ProfileFT = ValueFDB.getTable("港股-证券资料-东财")
ProfileData = ProfileFT.readData(
    factor_names=["证券代码", "证券简称"],
    ids=["700.HK", "2318.HK"],
    dts=[dt.datetime(2025, 1, 2)],
)
print(ProfileData.iloc[:, 0, :])

             证券代码  证券简称
700.HK   00700.HK  腾讯控股
2318.HK  02318.HK  中国平安


# WindDB 到 JYDB 转换

本章介绍一个完整的具体实现：以 WindDB 作为原生数据源，对外使用 JYDB 证券代码。调用方传入 JYDB ID，包装层转换为 WindDB ID 后读取数据，再把结果转换回 JYDB ID。

涉及三个专用对象：

| 对象 | 作用 |
|---|---|
| `JYDBWindDBHKStockIDMapping` | JYDB 港股代码到 WindDB 港股代码的映射 |
| `WindDBMutualFundIDMapping` | 公募基金外部代码到 Wind 内部基金代码的映射 |
| `JYDBWindDBFactorDB` | 包装 WindDB，并对外提供 JYDB 代码接口 |

## 港股代码映射

`JYDBWindDBHKStockIDMapping` 从两套数据库加载证券快照。匹配顺序为：

1. `ISIN_BASE`：ISIN 与基础代码一致；
2. `ISIN_MIGRATION`：识别相同 ISIN 下的代码迁移；
3. `FULL_CODE`：完整标准代码唯一一致且 ISIN 不冲突；
4. `BASE_LIST_DELIST`：基础代码、上市日期和退市日期一致；
5. `BASE_LIST`：基础代码和上市日期一致。

临时证券不会仅凭数据商 `!` 后缀猜测；双方 ISIN 非空且冲突时拒绝映射。`MappingInfo` 保存匹配方法、置信度及未映射原因。

In [2]:
import pandas as pd
from IPython.display import display

from QuantStudio.Factor.JYDB import JYDB
from QSExt.Factor.WindDB2 import WindDB2
from QSExt.Factor.JYDBWindDBIDMapping import (
    JYDBWindDBFactorDB,
    JYDBWindDBHKStockIDMapping,
    WindDBMutualFundIDMapping,
)

In [3]:
JYFDB = JYDB().connect()
WindFDB = WindDB2().connect()
AsOfDate = dt.date(2026, 11, 14)

hk_mapping = JYDBWindDBHKStockIDMapping(
    jydb=JYFDB,
    winddb=WindFDB,
    args={
        "AsOfDate": AsOfDate,
        "UnmappedPolicy": "empty",
    },
)
mapping_info = hk_mapping.MappingInfo
display(mapping_info[
    ["external_id", "windcode", "match_status",
     "match_method", "unmapped_reason"]
].head())

,external_id,windcode,match_status,match_method,unmapped_reason
0,00001.HK,0001.HK,MAPPED,ISIN_BASE,None
1,00002.HK,0002.HK,MAPPED,ISIN_BASE,None
2,00003.HK,0003.HK,MAPPED,ISIN_BASE,None
3,00004.HK,0004.HK,MAPPED,ISIN_BASE,None
4,00005.HK,0005.HK,MAPPED,ISIN_BASE,None


## 使用专用包装因子库

`JYDBWindDBFactorDB` 自动创建港股映射，并代理 WindDB 的 `getStockID`、`getMutualFundID`、`getFutureID`、`getOptionID`、`getIndustryID` 等库级接口。非港股库级 ID 原样保留。

两张 ETF 申购赎回表会自动使用 `WindDBMutualFundIDMapping`：

- `中国ETF申购赎回成份`；
- `中国ETF申购赎回清单`。

`JYDBWindDBFactorDB` 读取 WindDB2 的 `TableInfo.SecurityType`：`SecurityType == "港股"` 的表自动使用港股映射，`SecurityType == "公募基金"` 的表自动使用基金代码映射，其他表默认不转换。模块顶部的 `WINDDB_MANUAL_TABLES_BY_SECURITY_TYPE` 按证券类型集中配置手工覆盖，键为 `None` 时表示强制保持 WindDB 原生 ID；`WINDDB_MANUAL_ID_VALUE_FACTORS` 用 `{表名: {字段名: 证券类型}}` 为每个字段选择对应的映射。仍可通过构造参数或 `getTable` 参数覆盖。

In [4]:
FDB = JYDBWindDBFactorDB(
    jydb=JYFDB,
    winddb=WindFDB,
    as_of_date=AsOfDate,
    mapping_args={"UnmappedPolicy": "empty"},
    id_value_factors={},
    id_value_ambiguity="raise",
)

IDs = FDB.getStockID(
    exchange="HKEX",
    date=AsOfDate,
    is_current=False,
)
print(IDs[:5])

FT = FDB.getTable("香港股票日行情")
DTs = [dt.datetime(2026, 10, 12), dt.datetime(2026, 10, 13)]
Data = FT.readData(
    factor_names=["开盘价(元)", "收盘价(元)"],
    ids=IDs[:5],
    dts=DTs,
)
display(Data.iloc[0])

['00001.HK', '00002.HK', '00003.HK', '00004.HK', '00005.HK']


,00001.HK,00002.HK,00003.HK,00004.HK,00005.HK
2026-10-12,67.4,76.95,7.12,19.14,146.0
2026-10-13,66.2,77.25,7.12,18.73,148.0


## 单独配置 Wind 公募基金代码映射

`WindDBMutualFundIDMapping` 使用 `ChinaMutualFundDescription.s_info_outercode` 作为外部 ID，使用 `f_info_windcode` 作为 Wind 原生 ID。默认 `UnmappedPolicy="passthrough"`，与迁移前 WindDB2 的行为一致。

In [8]:
fund_mapping = FDB.MutualFundIDMapping
# FundIDs = WindFDB.getMutualFundID(is_current=True)[:5]
FundIDs = ["510050.OF", "159923.OF"]
fund_request = fund_mapping.resolve(FundIDs)
fund_mapping_result = pd.DataFrame({
    "ExternalID": fund_request.external_ids,
    "WindCode": fund_request.external_to_native,
})
display(fund_mapping_result)

ETFTable = FDB.getTable("中国ETF申购赎回清单")
print(type(ETFTable).__name__)

,ExternalID,WindCode
0,510050.OF,510050.SH
1,159923.OF,159923.SZ


IDMappingFactorTable


In [ ]:
FT = FDB.getTable("中国ETF申购赎回成份")
DTs = [dt.datetime(2026, 10, 12)]
Data = FT.readData(
    factor_names=["成份股Wind代码"],
    ids=["159519.OF"],
    dts=DTs,
)
display(Data.iloc[0])

## 运行时行为与注意事项

- `FactorNames`、表元数据和因子元数据由原生表提供；
- `getID` 返回反向映射后的外部 ID，允许多个外部 ID 对应一个原生 ID；
- `getDateTime(iid=...)` 会先把外部 ID 转成原生 ID；
- 直接 `readData`、`Engine`、`StackEngine` 和标准因子缓存均使用外部 ID 作为结果维度；
- 原生原始数据不进入包装表的 raw-data cache，避免两个 ID 命名空间混用；
- 映射对象按需加载数据库快照；调用 `refresh()` 可重新加载并更新 `MappingID`；
- `AsOfDate` 是映射快照截止日，不应晚于底层数据库可用数据日期；
- 对一对多的证券代码值，优先使用 `raise`，确认业务语义后再选择 `first` 或 `list`。

## 审计与排查

港股映射出现缺失或歧义时，先查看 `MappingInfo`。下面的代码汇总覆盖率和未映射原因。

常见原因包括 `NO_WIND_RECORD`、`AMBIGUOUS_CANDIDATES`、`ISIN_CONFLICT`、`TEMPORARY_SECURITY_NOT_RETAINED` 和 `INSUFFICIENT_IDENTIFIERS`。不要仅凭相同数字代码或数据商临时后缀强制建立映射。

In [6]:
mapping_info = FDB.IDMapping.MappingInfo
coverage = mapping_info["match_status"].value_counts(dropna=False)
unmapped = mapping_info.query("match_status == 'UNMAPPED'")
unmapped_reasons = unmapped["unmapped_reason"].value_counts(dropna=False)
display(coverage)
display(unmapped_reasons)

match_status
MAPPED      5387
UNMAPPED     859
Name: count, dtype: int64

unmapped_reason
AMBIGUOUS_CANDIDATES               848
TEMPORARY_SECURITY_NOT_RETAINED      4
NO_WIND_RECORD                       3
ISIN_CONFLICT                        3
INSUFFICIENT_IDENTIFIERS             1
Name: count, dtype: int64

In [ ]:
FDB.disconnect()
JYFDB.disconnect()